# Attention and Transformers, Explained Simply
Self-attention worked by hand, then a tiny attention model that understands "not", and PyTorch's built-in transformer layer. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
4,000 very short reviews (made up for this video), labelled positive (1) or negative (0). Half of them contain the word **not**, which flips the meaning.

### 4,000 short reviews
- `df.head(6)`: the first six reviews and their labels
- `str.contains(" not ")`: True for reviews that contain the word not

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "attention-and-transformers/short_reviews.csv")
df = pd.read_csv(url)
print(df.head(6))
print("share with 'not':", df["review"].str.contains(" not ").mean())

### Words to numbers
- `<pad>`: word number 0: padding, so every review is the same length
- `[0] * (L - len(ws))`: fill short reviews up with padding
- `has_not`: which test reviews contain not

In [ ]:
import torch

words = [r.split() for r in df["review"]]
vocab = ["<pad>"] + sorted({w for ws in words for w in ws})
index = {w: i for i, w in enumerate(vocab)}
L = max(len(ws) for ws in words)
X = torch.tensor([[index[w] for w in ws] + [0] * (L - len(ws)) for ws in words])
y = torch.tensor(df["positive"].values, dtype=torch.float32)
X_tr, y_tr, X_te, y_te = X[:3200], y[:3200], X[3200:], y[3200:]
has_not = df["review"][3200:].str.contains(" not ").values
print(len(vocab), "words; longest review:", L, "words")
print(df["review"][0], "->", X[0].tolist())

## 2. Attention by hand
Each word makes a **query** (what am I looking for?), a **key** (what do I offer?) and a **value** (what I'll pass on). **score = query . key / sqrt(d)**, **weights = softmax(scores)**, **new word = weights x values**.

### One word attends
- `q_good`: the query for the word good
- `keys`: keys for was, not, good
- `values`: values for was, not, good
- `/ math.sqrt(2)`: divide by the square root of the vector size, 2
- `softmax`: turn scores into weights that add up to 1
- `weights @ values`: good's new vector: mostly not's value

In [ ]:
import math

q_good = torch.tensor([1.0, 0.0])
keys = torch.tensor([[0.0, 1.0], [2.0, 0.0], [1.0, 1.0]])
values = torch.tensor([[0.0, 0.0], [-1.0, 0.0], [0.0, 1.0]])
scores = keys @ q_good / math.sqrt(2)
weights = scores.softmax(0)
print("scores: ", scores.round(decimals=3))
print("weights:", weights.round(decimals=3))
print("new good:", (weights @ values).round(decimals=3))

## 3. Without attention: averaging
A simple model: average the word embeddings of a review, then one layer decides. It sees which words are there, but not how they relate.

### Training and reporting
- `mean_of_words`: average the word vectors, ignoring padding
- `model(X_te) > 0`: a score above 0 means positive
- `the report`: accuracy on all test reviews, and separately with and without not

In [ ]:
import torch.nn as nn

def mean_of_words(h, mask):
    m = mask.float().unsqueeze(-1)
    return (h * m).sum(1) / m.sum(1)

def fit_and_report(model, epochs=30):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.BCEWithLogitsLoss()
    for epoch in range(epochs):
        for b in torch.randperm(len(X_tr)).split(64):
            optimizer.zero_grad()
            loss_fn(model(X_tr[b]), y_tr[b]).backward()
            optimizer.step()
    with torch.no_grad():
        right = ((model(X_te) > 0).float() == y_te).numpy()
    print(f"all: {right.mean():.3f}   without not: {right[~has_not].mean():.3f}"
          f"   with not: {right[has_not].mean():.3f}")

### The averaging model
- `nn.Embedding`: 16 learnable numbers per word
- `x != 0`: True for real words, False for padding
- `the output`: fine without not, but almost always wrong with not

In [ ]:
class Averager(nn.Module):
    def __init__(self, d=16):
        super().__init__()
        self.emb = nn.Embedding(len(vocab), d, padding_idx=0)
        self.out = nn.Linear(d, 1)
    def forward(self, x):
        h = mean_of_words(self.emb(x), x != 0)
        return self.out(h).squeeze(-1)

torch.manual_seed(0)
fit_and_report(Averager())

## 4. With self-attention
Before averaging, every word looks at every other word and mixes in what it needs. **Positions** are added so the model knows the word order.

### The attention model
- `self.pos`: a learnable vector for each position, added to the word
- `self.q, self.k, self.v`: layers that make each word's query, key and value
- `transpose(1, 2)`: so every query meets every key
- `masked_fill`: never attend to padding
- `h + self.weights @ self.v(h)`: each word plus what it gathered (a residual connection)

In [ ]:
class Attender(nn.Module):
    def __init__(self, d=16):
        super().__init__()
        self.emb = nn.Embedding(len(vocab), d, padding_idx=0)
        self.pos = nn.Embedding(L, d)
        self.q, self.k, self.v = [nn.Linear(d, d) for _ in range(3)]
        self.out = nn.Linear(d, 1)
    def forward(self, x):
        mask = x != 0
        h = self.emb(x) + self.pos(torch.arange(x.shape[1]))
        q, k = self.q(h), self.k(h)
        scores = q @ k.transpose(1, 2) / math.sqrt(q.shape[-1])
        scores = scores.masked_fill(~mask.unsqueeze(1), -1e9)
        self.weights = scores.softmax(-1)
        h = h + self.weights @ self.v(h)
        return self.out(mean_of_words(h, mask)).squeeze(-1)

### Train it
- `fit_and_report`: exactly the same training and test as the averaging model

In [ ]:
torch.manual_seed(0)
attender = Attender()
fit_and_report(attender)

### Look at the attention
- `attender.weights`: the attention weights saved during the last prediction
- `rows`: each word: where does it look?
- `the picture`: with not, the word good looks straight at not

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, text in zip(axes, ["the food was good", "the food was not good"]):
    ws = text.split()
    with torch.no_grad():
        attender(torch.tensor([[index[w] for w in ws]]))
    ax.imshow(attender.weights[0], cmap="Blues")
    ax.set_xticks(range(len(ws)), ws); ax.set_yticks(range(len(ws)), ws)
    ax.set_xlabel("looks at"); ax.set_title(text)
plt.show()

## 5. The full transformer layer
A real transformer layer adds **multi-head attention** (several attentions side by side), a small **feed-forward** network, and **layer normalisation**, each with a residual connection. PyTorch has it built in.

### PyTorch's transformer layer
- `nhead=4`: four attention heads, each looking for something different
- `dim_feedforward=32`: the small feed-forward network inside
- `the output`: same shape in, same shape out: layers can be stacked

In [ ]:
layer = nn.TransformerEncoderLayer(d_model=16, nhead=4, dim_feedforward=32,
                                   batch_first=True)
print("weights:", sum(p.numel() for p in layer.parameters()))
h = attender.emb(X_te[:2]) + attender.pos(torch.arange(L))
print(h.shape, "->", layer(h).shape)

## 6. Mini project: read new reviews
Score some reviews the model has never seen.

### Read new reviews
- `torch.sigmoid`: turn the score into a probability of positive
- `the output`: not bad is positive, not good is negative

In [ ]:
def read(text):
    x = torch.tensor([[index[w] for w in text.split()]])
    with torch.no_grad():
        p = torch.sigmoid(attender(x)).item()
    print(f"{p:.2f}  {'positive' if p > 0.5 else 'negative'}  <- {text}")

for text in ["the coffee was not bad", "honestly the music is really amazing",
             "i think the price was not good at all", "the room was terrible"]:
    read(text)

## Try it yourself
1. Work out the attention weights by hand if the key for "not" is [1, 0]. Then check them with the code.
2. Remove the positions (self.pos) from the attention model. Does it still work?
3. Type your own reviews into the mini project.